# PayShield AI — Feature Engineering

## AI-Powered Payment Success Optimization

### Objective

Transform raw payment transactions into time-window-based
monitoring features that can be used by machine learning models.

We will create features such as:

- Transaction volume
- Payment failure rate
- Timeout rate
- Average latency
- Maximum latency
- 95th percentile latency
- Bank error rate
- Transaction amount statistics
- Time-based features

The resulting dataset will represent the health of each
receiver bank over time.

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/raw/transactions.csv")

df["timestamp"] = pd.to_datetime(df["timestamp"])

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (100000, 12)


In [3]:
df.head()

,transaction_id,timestamp,amount,sender_bank,receiver_bank,payment_method,upi_app,bank_health,latency_ms,payment_status,timeout,error_code
0,TXN0000001,2026-01-01 00:01:39,345.00,AXIS,AXIS,UPI,PhonePe,NORMAL,1447.70,SUCCESS,0,NONE
1,TXN0000002,2026-01-01 00:07:00,2940.21,ICICI,KOTAK,UPI,BHIM,NORMAL,956.22,SUCCESS,0,NONE
2,TXN0000003,2026-01-01 00:17:34,219.04,INDUSIND,PNB,UPI,PhonePe,NORMAL,1132.68,SUCCESS,0,NONE
3,TXN0000004,2026-01-01 00:20:44,232.49,AXIS,INDUSIND,UPI,Paytm,NORMAL,1788.86,SUCCESS,0,NONE
4,TXN0000005,2026-01-01 00:35:07,590.03,INDUSIND,PNB,UPI,PhonePe,NORMAL,610.46,SUCCESS,0,NONE


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 12 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   transaction_id  100000 non-null  object        
 1   timestamp       100000 non-null  datetime64[ns]
 2   amount          100000 non-null  float64       
 3   sender_bank     100000 non-null  object        
 4   receiver_bank   100000 non-null  object        
 5   payment_method  100000 non-null  object        
 6   upi_app         100000 non-null  object        
 7   bank_health     100000 non-null  object        
 8   latency_ms      100000 non-null  float64       
 9   payment_status  100000 non-null  object        
 10  timeout         100000 non-null  int64         
 11  error_code      100000 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(8)
memory usage: 9.2+ MB


In [5]:
df["time_window"] = (
    df["timestamp"]
    .dt.floor("5min")
)

df[[
    "timestamp",
    "time_window"
]].head(10)

,timestamp,time_window
0,2026-01-01 00:01:39,2026-01-01 00:00:00
1,2026-01-01 00:07:00,2026-01-01 00:05:00
2,2026-01-01 00:17:34,2026-01-01 00:15:00
3,2026-01-01 00:20:44,2026-01-01 00:20:00
4,2026-01-01 00:35:07,2026-01-01 00:35:00
5,2026-01-01 00:37:54,2026-01-01 00:35:00
6,2026-01-01 00:38:59,2026-01-01 00:35:00
7,2026-01-01 00:51:05,2026-01-01 00:50:00
8,2026-01-01 00:52:17,2026-01-01 00:50:00
9,2026-01-01 00:53:00,2026-01-01 00:50:00


In [6]:
df["is_failed"] = (
    df["payment_status"] == "FAILED"
).astype(int)

df["is_timeout"] = (
    df["timeout"] == 1
).astype(int)

df["is_bank_error"] = (
    df["error_code"].isin([
        "BANK_TIMEOUT",
        "BANK_SERVER_ERROR"
    ])
).astype(int)

In [7]:
grouped = df.groupby([
    "receiver_bank",
    "time_window"
])

In [8]:
monitoring = (
    grouped
    .agg(
        transaction_count=(
            "transaction_id",
            "count"
        )
    )
    .reset_index()
)

monitoring.head()

,receiver_bank,time_window,transaction_count
0,AXIS,2026-01-01 00:00:00,1
1,AXIS,2026-01-01 00:55:00,2
2,AXIS,2026-01-01 01:00:00,1
3,AXIS,2026-01-01 01:25:00,1
4,AXIS,2026-01-01 01:35:00,1


In [9]:
failure_features = (
    grouped
    .agg(
        failure_rate=(
            "is_failed",
            "mean"
        )
    )
    .reset_index()
)

failure_features["failure_rate"] *= 100

failure_features.head()

,receiver_bank,time_window,failure_rate
0,AXIS,2026-01-01 00:00:00,0.0
1,AXIS,2026-01-01 00:55:00,0.0
2,AXIS,2026-01-01 01:00:00,0.0
3,AXIS,2026-01-01 01:25:00,0.0
4,AXIS,2026-01-01 01:35:00,0.0


In [10]:
timeout_features = (
    grouped
    .agg(
        timeout_rate=(
            "is_timeout",
            "mean"
        )
    )
    .reset_index()
)

timeout_features["timeout_rate"] *= 100

timeout_features.head()

,receiver_bank,time_window,timeout_rate
0,AXIS,2026-01-01 00:00:00,0.0
1,AXIS,2026-01-01 00:55:00,0.0
2,AXIS,2026-01-01 01:00:00,0.0
3,AXIS,2026-01-01 01:25:00,0.0
4,AXIS,2026-01-01 01:35:00,0.0


In [11]:
latency_features = (
    grouped["latency_ms"]
    .agg(
        avg_latency="mean",
        max_latency="max",
        p95_latency=lambda x: x.quantile(0.95)
    )
    .reset_index()
)

latency_features.head()

,receiver_bank,time_window,avg_latency,max_latency,p95_latency
0,AXIS,2026-01-01 00:00:00,1447.700,1447.70,1447.7000
1,AXIS,2026-01-01 00:55:00,881.935,883.95,883.7485
2,AXIS,2026-01-01 01:00:00,882.880,882.88,882.8800
3,AXIS,2026-01-01 01:25:00,751.670,751.67,751.6700
4,AXIS,2026-01-01 01:35:00,1161.920,1161.92,1161.9200


In [12]:
error_features = (
    grouped
    .agg(
        bank_error_rate=(
            "is_bank_error",
            "mean"
        )
    )
    .reset_index()
)

error_features["bank_error_rate"] *= 100

error_features.head()

,receiver_bank,time_window,bank_error_rate
0,AXIS,2026-01-01 00:00:00,0.0
1,AXIS,2026-01-01 00:55:00,0.0
2,AXIS,2026-01-01 01:00:00,0.0
3,AXIS,2026-01-01 01:25:00,0.0
4,AXIS,2026-01-01 01:35:00,0.0


In [15]:
grouped = df.groupby(
    ["receiver_bank", "time_window"]
)

print("Grouping created successfully!")

Grouping created successfully!


In [16]:
amount_features = (
    df.groupby(["receiver_bank", "time_window"])["amount"]
    .agg(
        avg_amount="mean",
        max_amount="max"
    )
    .reset_index()
)

print("Amount features created successfully!")
amount_features.head()

Amount features created successfully!


,receiver_bank,time_window,avg_amount,max_amount
0,AXIS,2026-01-01 00:00:00,345.00,345.00
1,AXIS,2026-01-01 00:55:00,244.72,320.59
2,AXIS,2026-01-01 01:00:00,321.00,321.00
3,AXIS,2026-01-01 01:25:00,395.70,395.70
4,AXIS,2026-01-01 01:35:00,32.14,32.14


In [17]:
print(amount_features.columns)

Index(['receiver_bank', 'time_window', 'avg_amount', 'max_amount'], dtype='object')


In [18]:
monitoring = (
    monitoring
    .merge(
        failure_features,
        on=["receiver_bank", "time_window"]
    )
    .merge(
        timeout_features,
        on=["receiver_bank", "time_window"]
    )
    .merge(
        latency_features,
        on=["receiver_bank", "time_window"]
    )
    .merge(
        error_features,
        on=["receiver_bank", "time_window"]
    )
    .merge(
        amount_features,
        on=["receiver_bank", "time_window"]
    )
)

In [19]:
monitoring.head()

,receiver_bank,time_window,transaction_count,failure_rate,timeout_rate,avg_latency,max_latency,p95_latency,bank_error_rate,avg_amount,max_amount
0,AXIS,2026-01-01 00:00:00,1,0.0,0.0,1447.700,1447.70,1447.7000,0.0,345.00,345.00
1,AXIS,2026-01-01 00:55:00,2,0.0,0.0,881.935,883.95,883.7485,0.0,244.72,320.59
2,AXIS,2026-01-01 01:00:00,1,0.0,0.0,882.880,882.88,882.8800,0.0,321.00,321.00
3,AXIS,2026-01-01 01:25:00,1,0.0,0.0,751.670,751.67,751.6700,0.0,395.70,395.70
4,AXIS,2026-01-01 01:35:00,1,0.0,0.0,1161.920,1161.92,1161.9200,0.0,32.14,32.14


In [20]:
monitoring["hour"] = (
    monitoring["time_window"].dt.hour
)

monitoring["day_of_week"] = (
    monitoring["time_window"].dt.dayofweek
)

monitoring["is_weekend"] = (
    monitoring["day_of_week"] >= 5
).astype(int)

In [21]:
health_features = (
    grouped["bank_health"]
    .agg(
        bank_health=lambda x: x.mode()[0]
    )
    .reset_index()
)

health_features.head()

,receiver_bank,time_window,bank_health
0,AXIS,2026-01-01 00:00:00,NORMAL
1,AXIS,2026-01-01 00:55:00,NORMAL
2,AXIS,2026-01-01 01:00:00,NORMAL
3,AXIS,2026-01-01 01:25:00,NORMAL
4,AXIS,2026-01-01 01:35:00,NORMAL


In [22]:
monitoring = monitoring.merge(
    health_features,
    on=["receiver_bank", "time_window"]
)

In [23]:
print("Monitoring dataset shape:")
print(monitoring.shape)

print("\nColumns:")
print(monitoring.columns.tolist())

Monitoring dataset shape:
(50984, 15)

Columns:
['receiver_bank', 'time_window', 'transaction_count', 'failure_rate', 'timeout_rate', 'avg_latency', 'max_latency', 'p95_latency', 'bank_error_rate', 'avg_amount', 'max_amount', 'hour', 'day_of_week', 'is_weekend', 'bank_health']


In [24]:
print(
    monitoring["bank_health"].value_counts()
)

bank_health
NORMAL      50611
SEVERE        187
RECOVERY      101
DEGRADED       85
Name: count, dtype: int64


In [25]:
monitoring.groupby("bank_health")[
    [
        "transaction_count",
        "failure_rate",
        "timeout_rate",
        "avg_latency",
        "p95_latency",
        "bank_error_rate"
    ]
].mean()

,transaction_count,failure_rate,timeout_rate,avg_latency,p95_latency,bank_error_rate
bank_health,,,,,,
DEGRADED,1.988235,19.756303,7.549020,2348.230488,2654.811118,9.117647
NORMAL,1.959930,2.442351,2.012888,1117.681362,1274.636620,1.256980
RECOVERY,2.376238,10.709571,4.059406,1689.765220,2018.505119,5.346535
SEVERE,2.122995,52.941176,35.248281,4055.891484,4734.420099,25.407436


In [26]:
monitoring.to_csv(
    "../data/processed/payment_monitoring_features.csv",
    index=False
)

print(
    "Feature dataset saved successfully!"
)

Feature dataset saved successfully!
